Question:
Explain what each of these status codes means and what the script must do
differently in response to each one. Then describe how a try-except-finally block around the
file-write step would have prevented the corrupted output file — specifically what finally
guarantees and why it runs even when an exception is raised.

200 → Success ✅

The request worked.

200 OK

Example:

Python → API
       ↓
API successfully processes request
       ↓
200

Your program can safely process the response.

if response.status_code == 200:
    print("Request successful")
401 → Unauthorized 🔐

This usually means the request does not have valid authentication credentials.

For example:

Wrong API key
Missing API key
Invalid token
Expired token

Your program should not keep sending the same request repeatedly.

It should:

check the API key/token
refresh authentication if appropriate
report an authentication failure

Example:

if response.status_code == 401:
    print("Authentication failed")
429 → Too Many Requests ⏳

This means you have sent too many requests in a certain amount of time.

For example:

Your program:
Request 1
Request 2
Request 3
...
Request 1000
       ↓
API says: STOP!
       ↓
429

Your program should wait and retry later, usually using a delay/backoff. If the API provides a Retry-After header, the client should follow it.

Example:

if response.status_code == 429:
    print("Too many requests. Wait and retry.")
500 → Internal Server Error 💥

This means something went wrong on the API server side.

It's generally not a problem with your Python syntax or request alone.

Example:

Python Backend
      ↓
Payment API
      ↓
Server problem
      ↓
500

Your program should:

handle the error safely
avoid assuming a valid response body
optionally retry if the operation is safe to retry
log the failure
avoid corrupting files or other output

Example:

if response.status_code == 500:
    print("Server error. Try again later.")

Now understand the file problem

Imagine your program does:

file = open("output.txt", "w")

file.write("Payment data")
file.write("More payment data")

Suppose an error happens here:

file.write(data)

The program may stop before the operation is properly completed.

That's why we use:

try
except
finally

What does try do?

Put the risky file operation inside try:

try:
    file.write(data)

Python tries to execute it.

If everything works:

try
 ↓
write successfully
 ↓
continue

If an exception occurs:

try
 ↓
ERROR ❌
 ↓
except

What does except do?

except handles the error.

try:
    file.write(data)

except Exception as e:
    print("Error while writing:", e)

Instead of the whole program unexpectedly crashing, you can handle the problem.


What does finally do?

This is the most important part of the question.

finally always runs, whether an exception occurs or not.

Example:

try:
    file.write(data)

except Exception as e:
    print("Error:", e)

finally:
    file.close()
If everything works:

try
 ↓
write
 ↓
finally
 ↓
close file

If an exception occurs:

try
 ↓
ERROR ❌
 ↓
except
 ↓
finally
 ↓
close file

So finally guarantees that the cleanup code runs even when an exception is raised.

Complete example

A simple version:

file = open("output.txt", "w")

try:
    file.write("Payment response data")

except Exception as e:
    print("Error:", e)

finally:
    file.close()
    print("File closed")

The important thing is:

finally:
    file.close()

Even if file.write() raises an exception, Python still executes the finally block

In [2]:
file = open("output.txt", "w")

try:
    file.write("Payment response data")

except Exception as e:
    print("Error:", e)

finally:
    file.close()
    print("File closed")

File closed
